# Edge-AI Road Hazard Detection — Phase 2 & 3: YOLOv8 Training + Jetson Nano Export

This notebook covers:
- **Phase 2**: Multi-dataset training on Google Colab GPU (Tesla T4) using Ultralytics YOLOv8s
- **Phase 3**: Evaluation (mAP@50: 92.6%, Recall: 85.3%) and ONNX / TensorRT export for NVIDIA Jetson Nano deployment

In [ ]:
# 1. Verify GPU acceleration
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Install Ultralytics and export dependencies
!pip install -q ultralytics onnx onnxruntime onnxslim

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Train YOLOv8s on Road Hazard Dataset (50 Epochs)
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data='/content/merged_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs/detect',
    name='pfpd_yolov8s',
    save=True,
    plots=True,
    seed=42
)

In [ ]:
# 5. Export to ONNX for NVIDIA Jetson Nano TensorRT
best_model = YOLO('/content/runs/detect/pfpd_yolov8s/weights/best.pt')
onnx_file = best_model.export(format='onnx', opset=12, dynamic=False, imgsz=640)
print(f'Exported ONNX model: {onnx_file}')

### Deploying on NVIDIA Jetson Nano
Run the following command on your Jetson Nano to build the high-speed TensorRT FP16 engine:
```bash
/usr/src/tensorrt/bin/trtexec --onnx=best.onnx --saveEngine=best.engine --fp16
```